# Ejercicio: asistente de documentos con NVIDIA NIM y Gradio

**Tiempo:** 40–45 minutos  
**Grupos:** 3–4 personas

## Contexto

En los notebooks anteriores construimos chatbots que responden preguntas
generales. En este ejercicio construiremos un asistente que responda preguntas
sobre un documento específico: **“Attention Is All You Need”**
(Vaswani et al., 2017).

Inyectar un documento completo en el contexto permite construir un primer
prototipo de *document question answering*. Esto ayuda a entender la motivación
de **RAG**, aunque todavía no es RAG: aquí no existe recuperación de fragmentos.

## Objetivo

Construir una aplicación Gradio donde el usuario pueda:

1. Cargar o leer el PDF del paper.
2. Hacer preguntas sobre su contenido.
3. Obtener respuestas basadas exclusivamente en el documento.

## Lo que van a aprender

- Extracción de texto de PDF con `pypdf`.
- Uso de contexto externo en un `system` prompt.
- Consumo de NVIDIA NIM mediante una API compatible con OpenAI.
- Streaming e historial con Gradio.
- Limitaciones de este enfoque y motivación de RAG.

## Paso 0: instalación y configuración

### 0.1 Instala las dependencias

- `pypdf`: extracción del texto del PDF.
- `gradio`: interfaz web.
- `openai`: cliente compatible con NVIDIA NIM.
- `python-dotenv`: lectura de variables de entorno.

```bash
pip install -q --upgrade pypdf gradio openai python-dotenv
```

In [1]:
# Propósito: instalar las librerías que usa el notebook antes de importarlas.
# Se ejecuta una sola vez por entorno; si ya están instaladas, solo se actualizan.

# %pip instala en el mismo entorno (kernel) que ejecuta el notebook, a diferencia de !pip.
# -q = modo silencioso (menos texto); --upgrade = instala la versión más reciente.
# pypdf: lee PDFs | gradio: interfaz web | openai: cliente de la API | python-dotenv: lee el archivo .env
%pip install -q --upgrade pypdf gradio openai python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 0.2 Descarga el paper

Descarga el PDF abierto desde:

```text
https://arxiv.org/pdf/1706.03762
```

Guárdalo junto al notebook como `attention_is_all_you_need.pdf`.

### 0.3 Configura tus credenciales

Crea un archivo `.env` con tu API key de NVIDIA:

```text
NVIDIA_API_KEY="tu_key_aqui"
```

Puedes crearla en [NVIDIA Build](https://build.nvidia.com/settings/api-keys).

## Paso 1: extracción de texto del PDF

Lee el PDF y extrae su contenido como texto plano.

**Instrucciones:**

1. Importa `PdfReader` desde `pypdf`.
2. Crea `extract_text_from_pdf(pdf_path)`.
3. Abre el PDF e itera sobre `reader.pages`.
4. Concatena el resultado de `page.extract_text()`; contempla páginas sin texto.
5. Retorna el texto completo.
6. Prueba la función e imprime los primeros 500 caracteres.

> **Pista:** usa `page.extract_text() or ""` para evitar valores `None`.

In [2]:
# Propósito: convertir el PDF del paper en texto plano.
# El modelo de lenguaje solo entiende texto, no archivos PDF, por eso este paso va primero.

# PdfReader abre un PDF y da acceso a sus páginas una por una.
from pypdf import PdfReader

# Ruta relativa: el PDF debe estar en la misma carpeta que el notebook.
PDF_PATH = "attention_is_all_you_need.pdf"


def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)  # carga el PDF; reader.pages es la lista de páginas
    text = ""  # acumulador: aquí se va pegando el texto de cada página
    for page in reader.pages:  # recorre las páginas en orden (el paper tiene 15)
        # Algunas páginas (por ejemplo, solo imágenes) devuelven None
        # "or ''" cambia None por texto vacío para que la suma no falle; "\n" separa una página de la siguiente.
        text += (page.extract_text() or "") + "\n"
    return text  # un solo string con todo el contenido del paper


document_text = extract_text_from_pdf(PDF_PATH)  # texto completo que luego se le dará al modelo
# Muestra solo los primeros 500 caracteres para verificar que la extracción funcionó (título, autores).
print(document_text[:500])

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz 


## Paso 2: inicialización del cliente de NVIDIA NIM

**Instrucciones:**

1. Importa `os`, `load_dotenv` y `OpenAI`.
2. Carga las variables con `load_dotenv()`.
3. Verifica que `NVIDIA_API_KEY` esté disponible sin imprimirla.
4. Inicializa `OpenAI` con:
   - `base_url="https://integrate.api.nvidia.com/v1"`
   - `api_key=os.getenv("NVIDIA_API_KEY")`
5. Define:

```python
MODELO = "nvidia/nemotron-3-super-120b-a12b"
```

In [3]:
# Propósito: crear el "cliente", el objeto que envía las preguntas al modelo en los servidores de NVIDIA.
# La API key se lee del archivo .env para no escribirla en el código y no exponerla.

import os  # permite leer variables de entorno (valores de configuración del sistema)
from dotenv import load_dotenv  # carga las variables definidas en el archivo .env
from openai import OpenAI  # cliente oficial de OpenAI; NVIDIA NIM usa el mismo formato de API

# Lee el archivo .env de la carpeta actual y deja NVIDIA_API_KEY disponible en os.getenv.
load_dotenv()

# Solo se verifica que exista, sin imprimir la key
# os.getenv devuelve None si la variable no existe; así se detecta el error antes de llamar a la API.
if os.getenv("NVIDIA_API_KEY"):
    print("NVIDIA_API_KEY cargada correctamente")
else:
    print("Falta NVIDIA_API_KEY en el archivo .env")

# Se reutiliza la librería de OpenAI cambiando solo la dirección del servidor (base_url).
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",  # servidor de NVIDIA NIM (servicio que aloja los modelos)
    api_key=os.getenv("NVIDIA_API_KEY"),  # credencial que identifica al usuario ante NVIDIA
)

# Nemotron 3 Super: modelo de NVIDIA con 120 mil millones de parámetros, de los cuales solo ~12 mil millones
# se activan por cada token (arquitectura "mixture of experts"), lo que lo hace más rápido.
MODELO = "nvidia/nemotron-3-super-120b-a12b"

NVIDIA_API_KEY cargada correctamente


## Paso 3: función de chat con el documento

Construye un `system` prompt que incluya el texto del paper y obligue al modelo
a fundamentar sus respuestas en él.

**Instrucciones:**

1. Crea `build_system_prompt(document_text)` para:
   - definir al asistente como experto en el paper;
   - incluir el documento entre delimitadores claros;
   - exigir respuestas basadas únicamente en el documento;
   - indicar que responda “No encuentro esa información en el documento” cuando corresponda.

2. Crea `history_to_messages(history)` para convertir el historial de Gradio a
   mensajes con roles `user` y `assistant`.

3. Crea `chat_con_documento(message, history, document_text)` para:
   - iniciar `messages` con el `system` prompt;
   - agregar el historial y el mensaje actual;
   - llamar `client.chat.completions.create(..., stream=True)`;
   - acumular `chunk.choices[0].delta.content`;
   - usar `yield` para entregar la respuesta progresivamente.

> **Pista:** utiliza `temperature=1.0` y `top_p=0.95`, valores recomendados para
> Nemotron 3 Super.

In [4]:
# Propósito: definir la lógica del chat: las reglas para el modelo, la conversión del historial
# y la llamada a la API con respuesta progresiva (streaming).

# Construye el "system prompt": el primer mensaje, invisible para el usuario, que fija las reglas del asistente.
# Se pega el paper completo dentro del prompt para que el modelo lo use como única fuente.
# Los delimitadores <documento> ... </documento> marcan dónde empieza y termina el paper.
# La frase exacta "No encuentro esa información..." da una salida fija cuando la respuesta no está (reduce inventos).
# Nota: no se ponen comentarios dentro del f-string porque se volverían parte del texto enviado al modelo.
def build_system_prompt(document_text):
    return f"""Eres un asistente experto en el paper "Attention Is All You Need" (Vaswani et al., 2017).
Responde ÚNICAMENTE con la información del documento que aparece entre <documento> y </documento>.
No uses tu conocimiento general.
Si la respuesta no está en el documento, responde exactamente: "No encuentro esa información en el documento."
Responde en el mismo idioma de la pregunta.

<documento>
{document_text}
</documento>"""


# Convierte el historial que guarda Gradio al formato que espera la API: [{"role": ..., "content": texto}].
# Es necesario porque el modelo no tiene memoria: en cada pregunta hay que reenviarle la conversación previa.
def history_to_messages(history):
    messages = []
    for item in history:  # cada item es un mensaje anterior, del usuario ("user") o del modelo ("assistant")
        content = item["content"]
        # Gradio 6 puede entregar el contenido como lista de bloques {"type": "text", "text": ...}
        if isinstance(content, list):  # caso lista de bloques: se unen solo las partes de texto
            text = ""
            for part in content:
                # Se ignoran bloques que no son texto (por ejemplo, archivos), porque la API solo recibe texto aquí.
                if isinstance(part, dict) and part.get("type") == "text":
                    text += part["text"]
            content = text
        messages.append({"role": item["role"], "content": content})  # conserva el rol original
    return messages


# Función que Gradio llama en cada pregunta. Recibe: el mensaje nuevo, el historial y el texto del paper.
def chat_con_documento(message, history, document_text):
    # Orden de la lista: 1) reglas + paper (system), 2) conversación previa, 3) pregunta actual (user).
    messages = [{"role": "system", "content": build_system_prompt(document_text)}]
    messages.extend(history_to_messages(history))  # extend agrega todos los elementos de la lista
    messages.append({"role": "user", "content": message})

    stream = client.chat.completions.create(
        model=MODELO,  # modelo definido en el paso 2
        messages=messages,  # toda la conversación, incluido el paper
        temperature=1.0,  # controla la aleatoriedad: 0 = siempre la palabra más probable; 1.0 = valor recomendado para Nemotron 3 Super
        top_p=0.95,  # solo elige entre las palabras que suman el 95 % de la probabilidad; descarta las muy improbables
        max_tokens=4096,  # límite de tokens (trozos de palabra) que puede generar la respuesta; evita respuestas sin fin
        stream=True,  # la respuesta llega por partes (chunks) en vez de toda al final
    )

    respuesta = ""  # acumulador del texto que se va recibiendo
    for chunk in stream:  # cada chunk trae un pedazo pequeño de la respuesta
        # Algunos chunks llegan sin texto (por ejemplo, el último)
        # delta.content = el texto nuevo de este chunk; se revisa que exista antes de sumarlo.
        if chunk.choices and chunk.choices[0].delta.content:
            respuesta += chunk.choices[0].delta.content
            # yield convierte la función en generador: entrega la respuesta parcial y continúa,
            # así Gradio muestra el texto escribiéndose en tiempo real.
            yield respuesta

## Paso 4: interfaz Gradio

Construye la interfaz con `gr.ChatInterface`.

**Instrucciones:**

1. Extrae el texto del PDF con la función del paso 1.
2. Muestra cuántas páginas y caracteres se extrajeron.
3. Configura `gr.ChatInterface` con:
   - `fn=chat_con_documento`;
   - un título y una descripción;
   - un `gr.Textbox` oculto dentro de `additional_inputs` para enviar el documento;
   - al menos tres preguntas de ejemplo.

```python
gr.Textbox(value=document_text, visible=False)
```

4. Lanza la interfaz:

```python
demo.launch(server_name="0.0.0.0", server_port=8080, show_error=True)
```

In [5]:
# Propósito: crear y lanzar la interfaz web de chat que conecta al usuario con chat_con_documento.
# Antes reporta el tamaño del documento, porque ese texto se envía completo en cada pregunta.

import gradio as gr  # librería para crear interfaces web con pocas líneas de Python

document_text = extract_text_from_pdf(PDF_PATH)  # función del paso 1
num_paginas = len(PdfReader(PDF_PATH).pages)  # número de páginas del PDF (15 en este paper)
print(f"Páginas extraídas: {num_paginas}")
# Cantidad de caracteres que se envían al modelo en cada pregunta (≈39 600 aquí).
# No es igual al número de tokens: un token suele ser un trozo de palabra (en inglés, ~4 caracteres).
print(f"Caracteres extraídos: {len(document_text)}")

# ChatInterface arma una ventana de chat completa (caja de texto, historial, botón de enviar).
demo = gr.ChatInterface(
    fn=chat_con_documento,  # función que responde; Gradio le pasa (mensaje, historial, entradas adicionales)
    title="Asistente: Attention Is All You Need",
    description="Haz preguntas sobre el paper. Las respuestas se basan solo en el documento.",
    # Textbox oculto: no se ve en pantalla, pero Gradio envía su valor (el paper) como tercer argumento de fn.
    additional_inputs=[gr.Textbox(value=document_text, visible=False)],
    examples=[  # preguntas de ejemplo que el usuario puede pulsar para probar rápido
        ["¿Cuál es la arquitectura principal propuesta en el paper?"],
        ["¿Qué es el mecanismo de atención?"],
        ["¿Cuántas capas tiene el encoder del modelo base?"],
    ],
)

# server_name="0.0.0.0": acepta conexiones desde cualquier equipo de la red, no solo desde este computador.
# server_port=8080: puerto donde queda la app (se abre en http://localhost:8080).
# show_error=True: muestra los errores en la interfaz, útil para depurar.
demo.launch(server_name="0.0.0.0", server_port=8080, show_error=True)

c:\Users\Emmanuel\Documents\ENTREGABLES IA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Páginas extraídas: 15
Caracteres extraídos: 39600
* Running on local URL:  http://0.0.0.0:8080
* To create a public link, set `share=True` in `launch()`.


## Paso 5: Prueba y reflexión
 
Una vez que la interfaz esté funcionando, prueba estas preguntas:
 
1. *"¿Cuál es la arquitectura principal propuesta en el paper?"*
2. *"¿Qué es el mecanismo de atención?"*
3. *"¿Cuántas capas tiene el encoder del modelo base?"*
4. *"¿Quiénes son los autores del paper?"*
5. *"¿Cuál es el resultado del modelo en la tarea WMT 2014 English-to-German?"*
 
Y esta pregunta trampa:
6. *"¿Qué es GPT-4?"*
 
Esta última pregunta **no está en el paper**. Observa cómo responde el modelo.
¿Usa su conocimiento general o respeta la instrucción de ceñirse al documento?
 

**Respuesta (resultado observado al probar):**
El modelo respondió las cinco preguntas con datos del paper: el **Transformer**, la atención como una suma ponderada de valores según qué tanto se parece la consulta a cada clave, **6 capas** en el encoder, los **8 autores** y **28.4 BLEU** (métrica que compara la traducción con una de referencia) para el Transformer *big*.
Ante la pregunta trampa sobre GPT-4 respondió *"No encuentro esa información en el documento."*: respetó el `system` prompt (el mensaje inicial con las reglas del asistente) y no usó su conocimiento general.
Como `temperature=1.0` agrega aleatoriedad, la redacción puede variar entre ejecuciones.

## Paso 6 (adicional): mejora el sistema

Si terminan antes, implementen una mejora:

**A) Medición del contexto**  
Reporta la cantidad de caracteres o estima los tokens del documento. Explica
por qué caracteres y tokens no son equivalentes.

**B) Subida dinámica de PDF**  
Agrega `gr.File` para cargar cualquier PDF y extrae el texto dentro del flujo de
la aplicación.

**C) Evidencia y páginas**  
Conserva separadores de página al extraer el PDF y exige que cada respuesta cite
la página que contiene la evidencia.

**D) Primer paso hacia RAG**  
Divide el documento en fragmentos y selecciona los más relevantes mediante una
búsqueda simple por palabras antes de llamar al modelo.

## Reflexión final

Discutan en grupo:

1. ¿Cuál es la limitación de enviar siempre el documento completo?
2. ¿Por qué este ejercicio todavía no implementa RAG?
3. ¿Cómo comprobarían que la respuesta proviene del documento y no del
   conocimiento previo del modelo?
4. ¿Qué riesgos aparecen al insertar directamente documentos de usuarios en el
   prompt?

## Recursos

- [pypdf](https://pypdf.readthedocs.io)
- [Gradio](https://www.gradio.app/docs)
- [NVIDIA Build](https://build.nvidia.com)
- [Nemotron 3 Super](https://build.nvidia.com/nvidia/nemotron-3-super-120b-a12b)
- [Paper original](https://arxiv.org/abs/1706.03762)

**Respuestas:**

1. **Limitación:** en cada pregunta se reenvía el paper completo (≈39 600 caracteres). Eso aumenta el costo y el tiempo de respuesta, y un documento más grande no cabría en la *ventana de contexto* (el máximo de texto que el modelo puede leer a la vez).
2. **No es RAG:** RAG (*Retrieval-Augmented Generation*, generación aumentada con recuperación) primero **busca** solo los fragmentos relevantes y luego genera. Aquí no se divide el documento ni se busca nada: siempre se envía todo.
3. **Comprobar el origen:** hacer preguntas cuya respuesta no está en el paper (como la de GPT-4) y verificar que responda "No encuentro esa información"; además, pedir que cite la página o la frase exacta y revisarla en el PDF.
4. **Riesgos:** *prompt injection* (el documento puede traer instrucciones ocultas, como "ignora tus reglas", que el modelo obedezca); envío de datos privados a un servicio externo; y documentos muy grandes que disparan el costo o superan el contexto.